In [1]:
import mygene
import pandas as pd
from tqdm import tqdm

In [2]:
import numpy as np
tf = pd.read_csv('/12tb_dsk2/danish/Gene_Regulatory_Network_ROSMAP/code_2/New_GRN_ROSMAP/Data/GRN_input/Raw/TF_Genes.txt', sep = '\t')
tf_list = tf['TF'].tolist()
tf_list = list(set(tf_list))
tf_list[0:10]

['ONECUT3',
 'ZNF302',
 'AR',
 'ETS1',
 'SOHLH2',
 'ZNF417',
 'MYF6',
 'ZNF410',
 'BPTF',
 'POU4F2']

In [3]:
tf

,TF,Genes,Edge
0,ATF7-NPFF,ENSG00000228327,1
1,ATF7-NPFF,ENSG00000237491,1
2,ATF7-NPFF,ENSG00000188290,1
3,ATF7-NPFF,ENSG00000187608,1
4,ATF7-NPFF,ENSG00000234396,1
...,...,...,...
2181580,ZSCAN9,ENSG00000185022,1
2181581,ZSCAN9,ENSG00000100417,1
2181582,ZSCAN9,ENSG00000280199,1
2181583,ZSCAN9,ENSG00000260613,1


In [4]:
count_matrix_mrn = pd.read_csv('../../data_preprocessed/count_matrix_preprocessed/final_count_matrix_mrn_normalized_for_grn.csv'
                              ,index_col=0)
count_matrix_mrn

,ENSG00000101901,ENSG00000174839,ENSG00000151466,ENSG00000168288,ENSG00000112343,ENSG00000172890,ENSG00000150991,ENSG00000100029,ENSG00000181894,ENSG00000206560,...,ENSG00000039560,ENSG00000223496,ENSG00000166681,ENSG00000078061,ENSG00000272143,ENSG00000120457,ENSG00000100528,ENSG00000116205,ENSG00000100603,ENSG00000114933
23_120411,483.749233,436.934791,240.759987,323.242575,144.901844,841.545325,3651.526471,648.714410,336.618130,771.323662,...,151.589621,1781.178052,5719.164322,404.610534,55.731478,54.616849,737.884775,458.112753,590.753672,407.954423
591_120522,428.287142,337.803943,230.947593,372.273733,135.293926,727.312570,5534.986540,669.575672,448.107271,754.888402,...,164.593248,2343.083980,6652.669483,354.177093,40.502003,24.128853,780.740745,437.766334,558.410599,388.646883
691_120605,393.796510,359.617945,187.239095,377.450240,148.602457,870.067384,6167.001950,703.632632,390.081449,777.933860,...,124.083051,2026.937508,7035.583309,420.544952,54.982909,74.301228,881.212568,374.478191,537.940893,349.215773
588_120522,348.370890,406.274499,221.172800,309.452072,123.401133,1071.691377,5986.853423,700.538739,355.015567,779.325616,...,127.198091,1990.555197,4542.110929,486.959855,86.380793,16.137071,673.960033,385.391230,557.203577,346.472412
604_120523,392.736981,331.265105,280.038543,409.812501,99.721042,727.417190,6091.179814,766.349378,376.344481,880.413857,...,152.313646,2573.622509,8597.866281,358.585939,45.762396,46.445417,754.055003,382.491668,627.013127,368.831251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677_120604,435.136655,397.298685,234.595414,402.343748,177.838459,804.687496,5738.758790,698.741180,345.586793,726.489025,...,187.928585,1928.475206,5934.254968,452.794375,47.928095,17.657719,853.876857,426.307796,568.830816,329.190339
142_120419,359.696638,334.004021,214.533352,340.427175,160.578856,772.063141,5559.882322,661.584888,416.220396,859.418039,...,196.548520,1418.232459,6881.767467,431.635966,35.969664,61.662281,709.116230,486.875092,503.575293,326.296236
120_120418,387.165725,399.624028,248.207730,270.249343,189.749539,1001.455898,4617.238772,798.289725,344.040830,795.414732,...,197.416187,2166.786397,4968.946250,382.374070,47.916550,25.874937,646.873427,485.873818,599.915208,390.999049
483_120515,454.480361,269.198605,171.157138,403.797907,105.519206,788.486037,6336.137532,806.764955,491.038196,825.874732,...,119.643824,2702.787227,6495.662632,317.388479,30.741816,39.881275,957.981455,432.878003,520.949152,334.836536


In [5]:


import pickle

# Replace 'your_file.pkl' with the path to your pickle file
with open('/12tb_dsk2/danish/Gene_Regulatory_Network_ROSMAP/code_2/New_GRN_ROSMAP/code/ensemble_to_gene_name.pickle', 'rb') as file:
    ensemble_to_gene_dict = pickle.load(file)

# Now, 'data' contains the object that was pickled
gene_to_ensemble = {v: k for k, v in ensemble_to_gene_dict.items()}

print(gene_to_ensemble['ZSCAN9'])

ENSG00000137185


In [6]:
x = list(set(tf.TF))
import requests
from tqdm import tqdm

def get_ensembl_ids_from_gene_names(gene_names):
    ensembl_ids = []

    for gene_name in tqdm(gene_names):
        # Ensembl REST API endpoint for gene name to ID mapping
        api_url = f'https://rest.ensembl.org/xrefs/symbol/human/{gene_name}?content-type=application/json'

        # Make a GET request to the Ensembl REST API
        response = requests.get(api_url)

        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            # Parse the JSON response
            data = response.json()

            # Check if there are results
            if data:
                # Append the first Ensemble ID found to the list
                ensembl_ids.append(data[0]['id'])
            else:
                # If no results, append None to the list
                ensembl_ids.append(None)
        else:
            # If the request was not successful, print an error message and append None to the list
            print(f"Error: Unable to retrieve data for gene {gene_name}. Status code: {response.status_code}")
            ensembl_ids.append(None)

    return ensembl_ids

# Example usage with a list of gene names
gene_names = x
ensembl_ids = get_ensembl_ids_from_gene_names(gene_names)

# Print the results


100%|█████████████████████████████████████████| 993/993 [03:35<00:00,  4.61it/s]


In [7]:
for gene_name, ensembl_id in zip(gene_names, ensembl_ids):
    if ensembl_id:
        print(f"The Ensemble ID for gene {gene_name} is: {ensembl_id}")
    else:
        print(f"No Ensemble ID found for gene {gene_name}")
    break

The Ensemble ID for gene ONECUT3 is: ENSG00000205922


In [8]:
dc = dict(zip(gene_names,ensembl_ids))

In [9]:
def return_gene_name(ensemble_idss):
    
    try:
        if dc[ensemble_idss] == None:
            return np.nan
        if dc[ensemble_idss] == 'Gene name not found':
            return np.nan

        return dc[ensemble_idss]
    except:
        return np.nan

tf['TF_ensemble'] = list(tf.TF.apply(return_gene_name))

In [10]:
tf

,TF,Genes,Edge,TF_ensemble
0,ATF7-NPFF,ENSG00000228327,1,ENSG00000267281
1,ATF7-NPFF,ENSG00000237491,1,ENSG00000267281
2,ATF7-NPFF,ENSG00000188290,1,ENSG00000267281
3,ATF7-NPFF,ENSG00000187608,1,ENSG00000267281
4,ATF7-NPFF,ENSG00000234396,1,ENSG00000267281
...,...,...,...,...
2181580,ZSCAN9,ENSG00000185022,1,ENSG00000137185
2181581,ZSCAN9,ENSG00000100417,1,ENSG00000137185
2181582,ZSCAN9,ENSG00000280199,1,ENSG00000137185
2181583,ZSCAN9,ENSG00000260613,1,ENSG00000137185


In [11]:
count_matrix = pd.read_csv('../../data_preprocessed/count_matrix_preprocessed/final_count_matrix_mrn_normalized_for_grn.csv',index_col=0).T
count_matrix

,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,...,65_120417,628_120524,587_120522,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,483.749233,428.287142,393.796510,348.370890,392.736981,315.365572,376.066735,301.448336,392.819935,409.181322,...,296.573693,299.072257,372.140831,354.778728,417.815236,435.136655,359.696638,387.165725,454.480361,377.965207
ENSG00000174839,436.934791,337.803943,359.617945,406.274499,331.265105,398.722111,390.872512,361.087396,369.115284,355.289148,...,329.526326,352.018064,357.304652,284.726877,323.876911,397.298685,334.004021,399.624028,269.198605,402.205256
ENSG00000151466,240.759987,230.947593,187.239095,221.172800,280.038543,286.190784,255.646415,215.784960,252.849614,192.947290,...,297.891798,197.473548,223.779038,245.181478,200.495231,234.595414,214.533352,248.207730,171.157138,249.582726
ENSG00000168288,323.242575,372.273733,377.450240,309.452072,409.812501,327.869053,325.727093,384.943019,436.842859,385.229245,...,338.753063,307.658064,247.269655,353.648860,339.299622,402.343748,340.427175,270.249343,403.797907,339.360685
ENSG00000112343,144.901844,135.293926,148.602457,123.401133,99.721042,168.102354,104.627490,111.687693,145.614286,124.417735,...,67.223370,114.477419,97.671514,61.012902,171.051875,177.838459,160.578856,189.749539,105.519206,320.507313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,54.616849,24.128853,74.301228,16.137071,46.445417,87.524366,23.689243,42.289515,29.348616,52.561503,...,27.680211,15.740645,17.308876,12.428554,29.443356,17.657719,61.662281,25.874937,39.881275,30.524506
ENSG00000100528,737.884775,780.740745,881.212568,673.960033,754.055003,608.502734,704.754983,720.006098,965.117945,926.146992,...,821.179603,891.492901,829.589694,767.180753,601.485693,853.876857,709.116230,646.873427,957.981455,716.428112
ENSG00000116205,458.112753,437.766334,374.478191,385.391230,382.491668,343.151085,419.497014,391.449098,516.987156,351.297135,...,369.069485,367.758709,589.738128,347.999517,487.918464,426.307796,486.875092,485.873818,432.878003,495.574333
ENSG00000100603,590.753672,558.410599,537.940893,557.203577,627.013127,594.609978,548.800799,590.968861,574.555595,544.244425,...,536.468858,569.525160,485.884873,595.440732,615.506338,568.830816,503.575293,599.915208,520.949152,565.601141


In [12]:
tf

,TF,Genes,Edge,TF_ensemble
0,ATF7-NPFF,ENSG00000228327,1,ENSG00000267281
1,ATF7-NPFF,ENSG00000237491,1,ENSG00000267281
2,ATF7-NPFF,ENSG00000188290,1,ENSG00000267281
3,ATF7-NPFF,ENSG00000187608,1,ENSG00000267281
4,ATF7-NPFF,ENSG00000234396,1,ENSG00000267281
...,...,...,...,...
2181580,ZSCAN9,ENSG00000185022,1,ENSG00000137185
2181581,ZSCAN9,ENSG00000100417,1,ENSG00000137185
2181582,ZSCAN9,ENSG00000280199,1,ENSG00000137185
2181583,ZSCAN9,ENSG00000260613,1,ENSG00000137185


In [13]:
tf_list = set(tf.TF_ensemble)
tf = tf[~tf.Genes.isin(tf_list)] # only to keep bipartite graph
tf 

,TF,Genes,Edge,TF_ensemble
0,ATF7-NPFF,ENSG00000228327,1,ENSG00000267281
1,ATF7-NPFF,ENSG00000237491,1,ENSG00000267281
3,ATF7-NPFF,ENSG00000187608,1,ENSG00000267281
4,ATF7-NPFF,ENSG00000234396,1,ENSG00000267281
5,ATF7-NPFF,ENSG00000171612,1,ENSG00000267281
...,...,...,...,...
2181579,ZSCAN9,ENSG00000237668,1,ENSG00000137185
2181581,ZSCAN9,ENSG00000100417,1,ENSG00000137185
2181582,ZSCAN9,ENSG00000280199,1,ENSG00000137185
2181583,ZSCAN9,ENSG00000260613,1,ENSG00000137185


In [14]:
 l = set(tf.TF_ensemble).union(set(tf.Genes))

In [15]:
remove_genes_tf = l - set(count_matrix.index)   # remove genes from tf-motif, which are not in count_matrix

In [16]:
tf = tf[~tf.Genes.isin(remove_genes_tf)]
tf = tf[~tf.TF_ensemble.isin(remove_genes_tf)]

In [17]:
tf

,TF,Genes,Edge,TF_ensemble
1151,AHR,ENSG00000224870,1,ENSG00000106546
1152,AHR,ENSG00000130770,1,ENSG00000106546
1153,AHR,ENSG00000116985,1,ENSG00000106546
1156,AHR,ENSG00000132854,1,ENSG00000106546
1157,AHR,ENSG00000122432,1,ENSG00000106546
...,...,...,...,...
2181563,ZSCAN9,ENSG00000160218,1,ENSG00000137185
2181567,ZSCAN9,ENSG00000131100,1,ENSG00000137185
2181572,ZSCAN9,ENSG00000178026,1,ENSG00000137185
2181577,ZSCAN9,ENSG00000185666,1,ENSG00000137185


In [18]:
 l = set(tf.TF_ensemble).union(set(tf.Genes))
l - set(count_matrix.index)

set()

In [19]:
len(set(count_matrix.index) - l)

157

In [20]:
tf

,TF,Genes,Edge,TF_ensemble
1151,AHR,ENSG00000224870,1,ENSG00000106546
1152,AHR,ENSG00000130770,1,ENSG00000106546
1153,AHR,ENSG00000116985,1,ENSG00000106546
1156,AHR,ENSG00000132854,1,ENSG00000106546
1157,AHR,ENSG00000122432,1,ENSG00000106546
...,...,...,...,...
2181563,ZSCAN9,ENSG00000160218,1,ENSG00000137185
2181567,ZSCAN9,ENSG00000131100,1,ENSG00000137185
2181572,ZSCAN9,ENSG00000178026,1,ENSG00000137185
2181577,ZSCAN9,ENSG00000185666,1,ENSG00000137185


 FLow Chart

In [21]:
# TF- motif and check aoverlapping with count matrix

In [22]:
tf_list = list(set(tf.TF))
tf_list[0:10]

['ZNF302',
 'ETS1',
 'ZNF417',
 'ZNF585A',
 'ZNF410',
 'BPTF',
 'HIVEP2',
 'ZNF44',
 'MTF1',
 'ZNF454']

In [23]:
len(tf_list)

715

In [24]:
import requests
import json

# Define your API key and transcription factor ID
#api_key = 'YOUR_API_KEY'
#tf_id = 'YOUR_TF_IDENTIFIER'

# Set the API endpoint
string_api_url = "https://version-12-0.string-db.org/api"
output_format = "tsv-no-header"
method = "interaction_partners"
# Define the query parameters
request_url = "/".join([string_api_url, output_format, method])

my_genes = tf_list


params = {

    "identifiers" : "%0d".join(my_genes), # your p'rotein
    "species" : 9606, # species NCBI identifier 
    "caller_identity" : "www.awesome_app.org", # your app name
    'required_score' : 500

}
response = requests.post(request_url, data=params)

##
## Read and parse the results
##


s = 0
query_name_list = []
partner_name_list = []
score = []
not_p = []
for line in tqdm(response.text.strip().split("\n")):
 
    l = line.strip().split("\t")
    query_ensp = l[0]
    query_name = l[2]
    if query_name not in tf_list:
        not_p.append(query_name)
        print(l)
       
        continue
        
    partner_ensp = l[1]
    partner_name = l[3]
    combined_score = l[5]

    ## print
    query_name_list.append(query_name)
    partner_name_list.append(partner_name)
    score.append(1)
    
    #print("\t".join([query_ensp, query_name, partner_name, combined_score]))

100%|█████████████████████████████████| 38655/38655 [00:00<00:00, 485791.05it/s]


In [25]:
set(not_p)
d = pd.DataFrame()
d['TF_name'] = query_name_list
d['Interacting_Partner'] = partner_name_list
d['Score'] = score
d

,TF_name,Interacting_Partner,Score
0,YBX2,SOHLH1,1
1,YBX2,NOBOX,1
2,YBX2,FIGLA,1
3,YBX2,LHX8,1
4,YBX2,RPS2,1
...,...,...,...
38650,RFX7,RFX8,1
38651,RFX7,ATR,1
38652,RFX7,DYRK1A,1
38653,RFX7,RFX4,1


In [26]:
d = d[d.Interacting_Partner.isin(set(d['TF_name']))]
len(set(d['TF_name']).intersection(set(d['Interacting_Partner'])))
d['TF_ensemble'] = list(d.TF_name.apply(return_gene_name))
d['Interacting_Partner_ensemble'] = list(d.Interacting_Partner.apply(return_gene_name))
d = d.dropna()

/tmp/ipykernel_81329/360976914.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d['TF_ensemble'] = list(d.TF_name.apply(return_gene_name))
/tmp/ipykernel_81329/360976914.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d['Interacting_Partner_ensemble'] = list(d.Interacting_Partner.apply(return_gene_name))


In [27]:
d = d.drop(columns = ['TF_name','Interacting_Partner'])
len(set(d['TF_ensemble']).union(set(d['Interacting_Partner_ensemble'])))  == len(set(d['TF_ensemble']))
len(set(d['TF_ensemble']).union(set(d['Interacting_Partner_ensemble'])))  == len(set(d['Interacting_Partner_ensemble']))
len(set(tf.TF))

715

In [28]:
d

,Score,TF_ensemble,Interacting_Partner_ensemble
12,1,ENSG00000006047,ENSG00000065978
15,1,ENSG00000065970,ENSG00000184486
17,1,ENSG00000065970,ENSG00000132005
23,1,ENSG00000070444,ENSG00000108788
24,1,ENSG00000070444,ENSG00000125952
...,...,...,...
38629,1,ENSG00000131408,ENSG00000112033
38645,1,ENSG00000185670,ENSG00000179456
38649,1,ENSG00000181827,ENSG00000143390
38653,1,ENSG00000181827,ENSG00000111783


In [29]:
len(set(d['TF_ensemble']).union(set(d['Interacting_Partner_ensemble'])))  == len(set(d['TF_ensemble']))

True

In [30]:
len(set(d['TF_ensemble']).union(set(d['Interacting_Partner_ensemble'])))  == len(set(d['Interacting_Partner_ensemble']))

True

In [31]:
l_ppi = set(d.TF_ensemble).union(set(d.Interacting_Partner_ensemble))
l_ppi - set(count_matrix.index)

set()

In [32]:
tf

,TF,Genes,Edge,TF_ensemble
1151,AHR,ENSG00000224870,1,ENSG00000106546
1152,AHR,ENSG00000130770,1,ENSG00000106546
1153,AHR,ENSG00000116985,1,ENSG00000106546
1156,AHR,ENSG00000132854,1,ENSG00000106546
1157,AHR,ENSG00000122432,1,ENSG00000106546
...,...,...,...,...
2181563,ZSCAN9,ENSG00000160218,1,ENSG00000137185
2181567,ZSCAN9,ENSG00000131100,1,ENSG00000137185
2181572,ZSCAN9,ENSG00000178026,1,ENSG00000137185
2181577,ZSCAN9,ENSG00000185666,1,ENSG00000137185


In [33]:
tf = tf[['TF_ensemble','Genes','Edge']]
tf.rename(columns = {'TF_ensemble':0, 'TF':1, 'Edge': 2}, inplace = True)

/tmp/ipykernel_81329/1665260759.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tf.rename(columns = {'TF_ensemble':0, 'TF':1, 'Edge': 2}, inplace = True)


In [36]:
tf.rename(columns = {'Genes':1}, inplace = True)
tf

/tmp/ipykernel_81329/1744024739.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tf.rename(columns = {'Genes':1}, inplace = True)


,0,1,2
1151,ENSG00000106546,ENSG00000224870,1
1152,ENSG00000106546,ENSG00000130770,1
1153,ENSG00000106546,ENSG00000116985,1
1156,ENSG00000106546,ENSG00000132854,1
1157,ENSG00000106546,ENSG00000122432,1
...,...,...,...
2181563,ENSG00000137185,ENSG00000160218,1
2181567,ENSG00000137185,ENSG00000131100,1
2181572,ENSG00000137185,ENSG00000178026,1
2181577,ENSG00000137185,ENSG00000185666,1


In [38]:
tf[2] = tf[2].astype(float)
tf

/tmp/ipykernel_81329/1301715162.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tf[2] = tf[2].astype(float)


,0,1,2
1151,ENSG00000106546,ENSG00000224870,1.0
1152,ENSG00000106546,ENSG00000130770,1.0
1153,ENSG00000106546,ENSG00000116985,1.0
1156,ENSG00000106546,ENSG00000132854,1.0
1157,ENSG00000106546,ENSG00000122432,1.0
...,...,...,...
2181563,ENSG00000137185,ENSG00000160218,1.0
2181567,ENSG00000137185,ENSG00000131100,1.0
2181572,ENSG00000137185,ENSG00000178026,1.0
2181577,ENSG00000137185,ENSG00000185666,1.0


In [39]:
count_matrix

,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,...,65_120417,628_120524,587_120522,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,483.749233,428.287142,393.796510,348.370890,392.736981,315.365572,376.066735,301.448336,392.819935,409.181322,...,296.573693,299.072257,372.140831,354.778728,417.815236,435.136655,359.696638,387.165725,454.480361,377.965207
ENSG00000174839,436.934791,337.803943,359.617945,406.274499,331.265105,398.722111,390.872512,361.087396,369.115284,355.289148,...,329.526326,352.018064,357.304652,284.726877,323.876911,397.298685,334.004021,399.624028,269.198605,402.205256
ENSG00000151466,240.759987,230.947593,187.239095,221.172800,280.038543,286.190784,255.646415,215.784960,252.849614,192.947290,...,297.891798,197.473548,223.779038,245.181478,200.495231,234.595414,214.533352,248.207730,171.157138,249.582726
ENSG00000168288,323.242575,372.273733,377.450240,309.452072,409.812501,327.869053,325.727093,384.943019,436.842859,385.229245,...,338.753063,307.658064,247.269655,353.648860,339.299622,402.343748,340.427175,270.249343,403.797907,339.360685
ENSG00000112343,144.901844,135.293926,148.602457,123.401133,99.721042,168.102354,104.627490,111.687693,145.614286,124.417735,...,67.223370,114.477419,97.671514,61.012902,171.051875,177.838459,160.578856,189.749539,105.519206,320.507313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,54.616849,24.128853,74.301228,16.137071,46.445417,87.524366,23.689243,42.289515,29.348616,52.561503,...,27.680211,15.740645,17.308876,12.428554,29.443356,17.657719,61.662281,25.874937,39.881275,30.524506
ENSG00000100528,737.884775,780.740745,881.212568,673.960033,754.055003,608.502734,704.754983,720.006098,965.117945,926.146992,...,821.179603,891.492901,829.589694,767.180753,601.485693,853.876857,709.116230,646.873427,957.981455,716.428112
ENSG00000116205,458.112753,437.766334,374.478191,385.391230,382.491668,343.151085,419.497014,391.449098,516.987156,351.297135,...,369.069485,367.758709,589.738128,347.999517,487.918464,426.307796,486.875092,485.873818,432.878003,495.574333
ENSG00000100603,590.753672,558.410599,537.940893,557.203577,627.013127,594.609978,548.800799,590.968861,574.555595,544.244425,...,536.468858,569.525160,485.884873,595.440732,615.506338,568.830816,503.575293,599.915208,520.949152,565.601141


In [41]:
ppi_df = d[['TF_ensemble','Interacting_Partner_ensemble','Score']]
ppi_df

,TF_ensemble,Interacting_Partner_ensemble,Score
12,ENSG00000006047,ENSG00000065978,1
15,ENSG00000065970,ENSG00000184486,1
17,ENSG00000065970,ENSG00000132005,1
23,ENSG00000070444,ENSG00000108788,1
24,ENSG00000070444,ENSG00000125952,1
...,...,...,...
38629,ENSG00000131408,ENSG00000112033,1
38645,ENSG00000185670,ENSG00000179456,1
38649,ENSG00000181827,ENSG00000143390,1
38653,ENSG00000181827,ENSG00000111783,1


In [42]:
ppi_df.rename(columns = {'TF_ensemble':0, 'Interacting_Partner_ensemble':1, 'Score': 2}, inplace = True)
ppi_df

,0,1,2
12,ENSG00000006047,ENSG00000065978,1
15,ENSG00000065970,ENSG00000184486,1
17,ENSG00000065970,ENSG00000132005,1
23,ENSG00000070444,ENSG00000108788,1
24,ENSG00000070444,ENSG00000125952,1
...,...,...,...
38629,ENSG00000131408,ENSG00000112033,1
38645,ENSG00000185670,ENSG00000179456,1
38649,ENSG00000181827,ENSG00000143390,1
38653,ENSG00000181827,ENSG00000111783,1


In [48]:
!ls

motif_ppi_preprocessing_and_panda.ipynb  netZooPy  netZooPy_dir


In [51]:
!ls ../../data_preprocessed/

count_matrix_preprocessed  temp_count_matrix_covariates_removal


In [61]:
!mkdir ../../data_preprocessed/grn_output/ad_nci_seperate_network/lioness

In [62]:
!ls

motif_ppi_preprocessing_and_panda.ipynb  netZooPy  netZooPy_dir


In [63]:
!mkdir ../../data_preprocessed/grn_input/

In [68]:
ppi_df.to_csv('../../data_preprocessed/grn_input/ppi.txt' ,sep = "\t",index=False, header=None)

In [69]:
ppi_data = pd.read_csv('../../data_preprocessed/grn_input/ppi.txt',header=None, sep = "\t")
ppi_data

,0,1,2
0,ENSG00000006047,ENSG00000065978,1
1,ENSG00000065970,ENSG00000184486,1
2,ENSG00000065970,ENSG00000132005,1
3,ENSG00000070444,ENSG00000108788,1
4,ENSG00000070444,ENSG00000125952,1
...,...,...,...
6977,ENSG00000131408,ENSG00000112033,1
6978,ENSG00000185670,ENSG00000179456,1
6979,ENSG00000181827,ENSG00000143390,1
6980,ENSG00000181827,ENSG00000111783,1


In [73]:
tf.to_csv('../../data_preprocessed/grn_input/tf.txt' ,sep = "\t",index=False, header=None)
tf

,0,1,2
1151,ENSG00000106546,ENSG00000224870,1.0
1152,ENSG00000106546,ENSG00000130770,1.0
1153,ENSG00000106546,ENSG00000116985,1.0
1156,ENSG00000106546,ENSG00000132854,1.0
1157,ENSG00000106546,ENSG00000122432,1.0
...,...,...,...
2181563,ENSG00000137185,ENSG00000160218,1.0
2181567,ENSG00000137185,ENSG00000131100,1.0
2181572,ENSG00000137185,ENSG00000178026,1.0
2181577,ENSG00000137185,ENSG00000185666,1.0


In [74]:
motif_data = pd.read_csv('../../data_preprocessed/grn_input/tf.txt' ,header=None, sep = "\t")
motif_data

,0,1,2
0,ENSG00000106546,ENSG00000224870,1.0
1,ENSG00000106546,ENSG00000130770,1.0
2,ENSG00000106546,ENSG00000116985,1.0
3,ENSG00000106546,ENSG00000132854,1.0
4,ENSG00000106546,ENSG00000122432,1.0
...,...,...,...
698912,ENSG00000137185,ENSG00000160218,1.0
698913,ENSG00000137185,ENSG00000131100,1.0
698914,ENSG00000137185,ENSG00000178026,1.0
698915,ENSG00000137185,ENSG00000185666,1.0


In [103]:
count_matrix.to_csv('../../data_preprocessed/grn_input/count_matrix_ad_nci.txt' ,sep = "\t")
count_matrix

,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,...,65_120417,628_120524,587_120522,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,483.749233,428.287142,393.796510,348.370890,392.736981,315.365572,376.066735,301.448336,392.819935,409.181322,...,296.573693,299.072257,372.140831,354.778728,417.815236,435.136655,359.696638,387.165725,454.480361,377.965207
ENSG00000174839,436.934791,337.803943,359.617945,406.274499,331.265105,398.722111,390.872512,361.087396,369.115284,355.289148,...,329.526326,352.018064,357.304652,284.726877,323.876911,397.298685,334.004021,399.624028,269.198605,402.205256
ENSG00000151466,240.759987,230.947593,187.239095,221.172800,280.038543,286.190784,255.646415,215.784960,252.849614,192.947290,...,297.891798,197.473548,223.779038,245.181478,200.495231,234.595414,214.533352,248.207730,171.157138,249.582726
ENSG00000168288,323.242575,372.273733,377.450240,309.452072,409.812501,327.869053,325.727093,384.943019,436.842859,385.229245,...,338.753063,307.658064,247.269655,353.648860,339.299622,402.343748,340.427175,270.249343,403.797907,339.360685
ENSG00000112343,144.901844,135.293926,148.602457,123.401133,99.721042,168.102354,104.627490,111.687693,145.614286,124.417735,...,67.223370,114.477419,97.671514,61.012902,171.051875,177.838459,160.578856,189.749539,105.519206,320.507313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,54.616849,24.128853,74.301228,16.137071,46.445417,87.524366,23.689243,42.289515,29.348616,52.561503,...,27.680211,15.740645,17.308876,12.428554,29.443356,17.657719,61.662281,25.874937,39.881275,30.524506
ENSG00000100528,737.884775,780.740745,881.212568,673.960033,754.055003,608.502734,704.754983,720.006098,965.117945,926.146992,...,821.179603,891.492901,829.589694,767.180753,601.485693,853.876857,709.116230,646.873427,957.981455,716.428112
ENSG00000116205,458.112753,437.766334,374.478191,385.391230,382.491668,343.151085,419.497014,391.449098,516.987156,351.297135,...,369.069485,367.758709,589.738128,347.999517,487.918464,426.307796,486.875092,485.873818,432.878003,495.574333
ENSG00000100603,590.753672,558.410599,537.940893,557.203577,627.013127,594.609978,548.800799,590.968861,574.555595,544.244425,...,536.468858,569.525160,485.884873,595.440732,615.506338,568.830816,503.575293,599.915208,520.949152,565.601141


In [104]:
exp_data = pd.read_csv('../../data_preprocessed/grn_input/count_matrix_ad_nci.txt', sep="\t", index_col=0)
exp_data 

,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,...,65_120417,628_120524,587_120522,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,483.749233,428.287142,393.796510,348.370890,392.736981,315.365572,376.066735,301.448336,392.819935,409.181322,...,296.573693,299.072257,372.140831,354.778728,417.815236,435.136655,359.696638,387.165725,454.480361,377.965207
ENSG00000174839,436.934791,337.803943,359.617945,406.274499,331.265105,398.722111,390.872512,361.087396,369.115284,355.289148,...,329.526326,352.018064,357.304652,284.726877,323.876911,397.298685,334.004021,399.624028,269.198605,402.205256
ENSG00000151466,240.759987,230.947593,187.239095,221.172800,280.038543,286.190784,255.646415,215.784960,252.849614,192.947290,...,297.891798,197.473548,223.779038,245.181478,200.495231,234.595414,214.533352,248.207730,171.157138,249.582726
ENSG00000168288,323.242575,372.273733,377.450240,309.452072,409.812501,327.869053,325.727093,384.943019,436.842859,385.229245,...,338.753063,307.658064,247.269655,353.648860,339.299622,402.343748,340.427175,270.249343,403.797907,339.360685
ENSG00000112343,144.901844,135.293926,148.602457,123.401133,99.721042,168.102354,104.627490,111.687693,145.614286,124.417735,...,67.223370,114.477419,97.671514,61.012902,171.051875,177.838459,160.578856,189.749539,105.519206,320.507313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,54.616849,24.128853,74.301228,16.137071,46.445417,87.524366,23.689243,42.289515,29.348616,52.561503,...,27.680211,15.740645,17.308876,12.428554,29.443356,17.657719,61.662281,25.874937,39.881275,30.524506
ENSG00000100528,737.884775,780.740745,881.212568,673.960033,754.055003,608.502734,704.754983,720.006098,965.117945,926.146992,...,821.179603,891.492901,829.589694,767.180753,601.485693,853.876857,709.116230,646.873427,957.981455,716.428112
ENSG00000116205,458.112753,437.766334,374.478191,385.391230,382.491668,343.151085,419.497014,391.449098,516.987156,351.297135,...,369.069485,367.758709,589.738128,347.999517,487.918464,426.307796,486.875092,485.873818,432.878003,495.574333
ENSG00000100603,590.753672,558.410599,537.940893,557.203577,627.013127,594.609978,548.800799,590.968861,574.555595,544.244425,...,536.468858,569.525160,485.884873,595.440732,615.506338,568.830816,503.575293,599.915208,520.949152,565.601141


# Splitting AD and NCI

In [105]:
exp_data = pd.read_csv('../../data_preprocessed/grn_input/count_matrix_ad_nci.txt', sep="\t", index_col=0)
exp_data 

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,...,65_120417,628_120524,587_120522,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,483.749233,428.287142,393.796510,348.370890,392.736981,315.365572,376.066735,301.448336,392.819935,409.181322,...,296.573693,299.072257,372.140831,354.778728,417.815236,435.136655,359.696638,387.165725,454.480361,377.965207
ENSG00000174839,436.934791,337.803943,359.617945,406.274499,331.265105,398.722111,390.872512,361.087396,369.115284,355.289148,...,329.526326,352.018064,357.304652,284.726877,323.876911,397.298685,334.004021,399.624028,269.198605,402.205256
ENSG00000151466,240.759987,230.947593,187.239095,221.172800,280.038543,286.190784,255.646415,215.784960,252.849614,192.947290,...,297.891798,197.473548,223.779038,245.181478,200.495231,234.595414,214.533352,248.207730,171.157138,249.582726
ENSG00000168288,323.242575,372.273733,377.450240,309.452072,409.812501,327.869053,325.727093,384.943019,436.842859,385.229245,...,338.753063,307.658064,247.269655,353.648860,339.299622,402.343748,340.427175,270.249343,403.797907,339.360685
ENSG00000112343,144.901844,135.293926,148.602457,123.401133,99.721042,168.102354,104.627490,111.687693,145.614286,124.417735,...,67.223370,114.477419,97.671514,61.012902,171.051875,177.838459,160.578856,189.749539,105.519206,320.507313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,54.616849,24.128853,74.301228,16.137071,46.445417,87.524366,23.689243,42.289515,29.348616,52.561503,...,27.680211,15.740645,17.308876,12.428554,29.443356,17.657719,61.662281,25.874937,39.881275,30.524506
ENSG00000100528,737.884775,780.740745,881.212568,673.960033,754.055003,608.502734,704.754983,720.006098,965.117945,926.146992,...,821.179603,891.492901,829.589694,767.180753,601.485693,853.876857,709.116230,646.873427,957.981455,716.428112
ENSG00000116205,458.112753,437.766334,374.478191,385.391230,382.491668,343.151085,419.497014,391.449098,516.987156,351.297135,...,369.069485,367.758709,589.738128,347.999517,487.918464,426.307796,486.875092,485.873818,432.878003,495.574333
ENSG00000100603,590.753672,558.410599,537.940893,557.203577,627.013127,594.609978,548.800799,590.968861,574.555595,544.244425,...,536.468858,569.525160,485.884873,595.440732,615.506338,568.830816,503.575293,599.915208,520.949152,565.601141


In [106]:
import pickle
with open('/12tb_dsk2/danish/Gene_Regulatory_Network_ROSMAP/code/clinical_data_pickle//NCI_with_No_Plaques.pkl','rb') as f:
    NCI_with_No_Plaques= pickle.load(f)

with open('/12tb_dsk2/danish/Gene_Regulatory_Network_ROSMAP/code/clinical_data_pickle/AD_with_Plaques.pkl','rb') as f:
    AD_with_Plaques= pickle.load(f)

with open('/12tb_dsk2/danish/Gene_Regulatory_Network_ROSMAP/code/clinical_data_pickle/AD_with_No_Plaques.pkl','rb') as f:
    AD_with_No_Plaques= pickle.load(f)

with open('/12tb_dsk2/danish/Gene_Regulatory_Network_ROSMAP/code/clinical_data_pickle/NCI_with_Plaques.pkl','rb') as f:
    NCI_with_Plaques = pickle.load(f)

In [109]:
nci_columns = set(NCI_with_No_Plaques).intersection(set(exp_data.columns))
ad_columns =  set(AD_with_Plaques).intersection(set(exp_data.columns))

In [113]:
nci_df = exp_data[list(nci_columns)]
nci_df 

,352_120501,938_131101,564_120517,142_120419,456_120507,950_131107,812_130702,360_120502,283_120430,340_120501,...,588_120522,213_120425,174_120424,04_120405,508_120515,633_120524,146_120419,345_120501,498_120515,152_120419
ENSG00000101901,381.584859,415.660727,400.729313,359.696638,412.832101,393.279400,425.798569,369.168485,354.778728,529.577639,...,348.370890,575.077436,377.965207,415.768825,438.602684,342.797302,482.021145,355.230899,313.789066,368.087123
ENSG00000174839,396.591005,312.375334,450.384902,334.004021,360.408977,349.925765,360.746010,324.205657,284.726877,387.495833,...,406.274499,338.872097,402.205256,345.078823,346.574443,365.947249,311.510264,332.206675,401.186691,329.468146
ENSG00000151466,229.379662,236.800657,266.572108,214.533352,221.979166,227.090467,198.114612,235.857643,245.181478,357.049732,...,221.172800,239.077136,249.582726,252.995795,165.455030,200.336085,196.743325,239.013383,286.411496,185.853826
ENSG00000168288,360.147507,372.835076,376.337094,340.427175,391.535207,341.667930,328.219730,331.305051,353.648860,382.882787,...,309.452072,374.051616,339.360685,412.048298,390.630516,365.947249,488.579256,381.544299,294.835364,399.465041
ENSG00000112343,201.511105,166.264291,153.322520,160.578856,145.801814,91.868416,266.124106,183.795421,61.012902,182.676607,...,123.401133,163.692454,320.507313,129.288292,121.398957,139.790068,93.999588,78.940200,164.265417,195.508570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,49.305909,31.489449,27.005671,61.662281,20.477783,26.837964,106.449642,29.975219,12.428554,47.053065,...,16.137071,51.692354,30.524506,32.554606,31.328763,16.917269,33.883573,17.542267,54.755139,20.516331
ENSG00000100528,767.457188,753.227621,595.867066,709.116230,670.033053,831.976893,796.893850,640.523099,767.180753,830.348214,...,673.960033,997.949608,716.428112,793.402254,982.939944,617.925525,920.321552,787.209216,572.822993,1000.472868
ENSG00000116205,503.777763,435.813974,360.656382,486.875092,408.736545,353.022453,419.884700,347.081482,347.999517,319.222758,...,385.391230,368.308021,495.574333,410.188035,367.133943,380.193371,415.347019,367.291208,391.709841,299.297070
ENSG00000100603,604.533315,554.214303,574.088299,503.575293,578.292586,506.824633,563.296024,539.553940,595.440732,572.940267,...,557.203577,473.128627,565.601141,547.847512,490.490948,694.498429,489.672275,685.244791,501.220119,539.458832


In [114]:
ad_df = exp_data[list(ad_columns)]
ad_df 

,202_120424,323_120501,550_120517,02_120405,424_120507,706_120605,104_120418,231_120425,34_120416,77_120417,...,326_120501,537_120516,628_120524,554_120517,76_120417,568_120521,711_120531,606_120523,689_120604,56_120417
ENSG00000101901,487.776747,352.457509,343.951603,349.968264,302.032190,430.056914,292.120027,339.554217,306.555120,218.653011,...,382.224858,353.368284,299.072257,347.460210,330.748677,369.362954,392.819935,349.680866,401.613563,241.233791
ENSG00000174839,386.519521,323.587269,350.534409,320.946506,402.162427,414.444287,354.717175,342.220874,408.740161,384.120155,...,375.746470,323.582452,352.018064,435.637424,364.929391,366.938252,369.115284,448.503719,360.067333,324.789629
ENSG00000151466,198.159304,228.556064,215.586890,341.432453,228.165622,252.640695,182.823418,227.554659,301.394260,191.075154,...,286.128778,217.978137,197.473548,237.238693,191.009874,172.962083,252.849614,235.654497,225.224303,311.312881
ENSG00000168288,394.141032,332.007756,347.243006,472.030366,336.503255,370.445065,338.819487,284.443324,237.399588,241.306251,...,399.500557,341.183171,307.658064,364.255870,363.924076,357.239443,436.842859,322.314537,348.405233,242.581466
ENSG00000112343,184.005068,122.698518,203.244129,118.647777,200.260474,178.835548,158.976885,121.777298,143.471925,207.818853,...,136.046136,292.442718,114.477419,124.917719,192.015189,178.619722,145.614286,115.546721,86.736868,164.416327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,62.060881,31.276093,106.147743,37.557570,39.395503,43.999222,46.699460,31.999874,56.769467,41.366786,...,16.195969,67.695074,15.740645,45.138335,58.308277,33.945829,29.348616,45.610548,28.426368,17.519773
ENSG00000100528,719.688460,632.739418,737.274249,739.201261,658.233196,755.083427,851.519942,696.886143,615.174585,684.521814,...,1099.166399,695.905356,891.492901,721.163639,707.741850,682.149525,965.117945,865.080056,758.036490,654.969958
ENSG00000116205,383.253159,419.821401,404.019706,370.454211,485.877871,476.894796,411.352691,331.554249,419.061882,373.285997,...,469.683088,523.959869,367.758709,420.941222,395.088845,472.816910,516.987156,460.666532,462.839587,404.302443
ENSG00000100603,543.304904,572.593086,552.132836,652.989566,600.781421,545.022624,602.124953,529.775690,528.472127,480.642655,...,651.077936,522.605968,569.525160,535.361653,499.641618,587.586143,574.555595,600.538879,546.660930,549.851323


In [115]:
ad_df.to_csv('../../data_preprocessed/grn_input/count_matrix_ad.txt' ,sep = "\t")
ad_df

,202_120424,323_120501,550_120517,02_120405,424_120507,706_120605,104_120418,231_120425,34_120416,77_120417,...,326_120501,537_120516,628_120524,554_120517,76_120417,568_120521,711_120531,606_120523,689_120604,56_120417
ENSG00000101901,487.776747,352.457509,343.951603,349.968264,302.032190,430.056914,292.120027,339.554217,306.555120,218.653011,...,382.224858,353.368284,299.072257,347.460210,330.748677,369.362954,392.819935,349.680866,401.613563,241.233791
ENSG00000174839,386.519521,323.587269,350.534409,320.946506,402.162427,414.444287,354.717175,342.220874,408.740161,384.120155,...,375.746470,323.582452,352.018064,435.637424,364.929391,366.938252,369.115284,448.503719,360.067333,324.789629
ENSG00000151466,198.159304,228.556064,215.586890,341.432453,228.165622,252.640695,182.823418,227.554659,301.394260,191.075154,...,286.128778,217.978137,197.473548,237.238693,191.009874,172.962083,252.849614,235.654497,225.224303,311.312881
ENSG00000168288,394.141032,332.007756,347.243006,472.030366,336.503255,370.445065,338.819487,284.443324,237.399588,241.306251,...,399.500557,341.183171,307.658064,364.255870,363.924076,357.239443,436.842859,322.314537,348.405233,242.581466
ENSG00000112343,184.005068,122.698518,203.244129,118.647777,200.260474,178.835548,158.976885,121.777298,143.471925,207.818853,...,136.046136,292.442718,114.477419,124.917719,192.015189,178.619722,145.614286,115.546721,86.736868,164.416327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,62.060881,31.276093,106.147743,37.557570,39.395503,43.999222,46.699460,31.999874,56.769467,41.366786,...,16.195969,67.695074,15.740645,45.138335,58.308277,33.945829,29.348616,45.610548,28.426368,17.519773
ENSG00000100528,719.688460,632.739418,737.274249,739.201261,658.233196,755.083427,851.519942,696.886143,615.174585,684.521814,...,1099.166399,695.905356,891.492901,721.163639,707.741850,682.149525,965.117945,865.080056,758.036490,654.969958
ENSG00000116205,383.253159,419.821401,404.019706,370.454211,485.877871,476.894796,411.352691,331.554249,419.061882,373.285997,...,469.683088,523.959869,367.758709,420.941222,395.088845,472.816910,516.987156,460.666532,462.839587,404.302443
ENSG00000100603,543.304904,572.593086,552.132836,652.989566,600.781421,545.022624,602.124953,529.775690,528.472127,480.642655,...,651.077936,522.605968,569.525160,535.361653,499.641618,587.586143,574.555595,600.538879,546.660930,549.851323


In [116]:
ad_data = pd.read_csv('../../data_preprocessed/grn_input/count_matrix_ad.txt', sep="\t", index_col=0)
ad_data 

,202_120424,323_120501,550_120517,02_120405,424_120507,706_120605,104_120418,231_120425,34_120416,77_120417,...,326_120501,537_120516,628_120524,554_120517,76_120417,568_120521,711_120531,606_120523,689_120604,56_120417
ENSG00000101901,487.776747,352.457509,343.951603,349.968264,302.032190,430.056914,292.120027,339.554217,306.555120,218.653011,...,382.224858,353.368284,299.072257,347.460210,330.748677,369.362954,392.819935,349.680866,401.613563,241.233791
ENSG00000174839,386.519521,323.587269,350.534409,320.946506,402.162427,414.444287,354.717175,342.220874,408.740161,384.120155,...,375.746470,323.582452,352.018064,435.637424,364.929391,366.938252,369.115284,448.503719,360.067333,324.789629
ENSG00000151466,198.159304,228.556064,215.586890,341.432453,228.165622,252.640695,182.823418,227.554659,301.394260,191.075154,...,286.128778,217.978137,197.473548,237.238693,191.009874,172.962083,252.849614,235.654497,225.224303,311.312881
ENSG00000168288,394.141032,332.007756,347.243006,472.030366,336.503255,370.445065,338.819487,284.443324,237.399588,241.306251,...,399.500557,341.183171,307.658064,364.255870,363.924076,357.239443,436.842859,322.314537,348.405233,242.581466
ENSG00000112343,184.005068,122.698518,203.244129,118.647777,200.260474,178.835548,158.976885,121.777298,143.471925,207.818853,...,136.046136,292.442718,114.477419,124.917719,192.015189,178.619722,145.614286,115.546721,86.736868,164.416327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,62.060881,31.276093,106.147743,37.557570,39.395503,43.999222,46.699460,31.999874,56.769467,41.366786,...,16.195969,67.695074,15.740645,45.138335,58.308277,33.945829,29.348616,45.610548,28.426368,17.519773
ENSG00000100528,719.688460,632.739418,737.274249,739.201261,658.233196,755.083427,851.519942,696.886143,615.174585,684.521814,...,1099.166399,695.905356,891.492901,721.163639,707.741850,682.149525,965.117945,865.080056,758.036490,654.969958
ENSG00000116205,383.253159,419.821401,404.019706,370.454211,485.877871,476.894796,411.352691,331.554249,419.061882,373.285997,...,469.683088,523.959869,367.758709,420.941222,395.088845,472.816910,516.987156,460.666532,462.839587,404.302443
ENSG00000100603,543.304904,572.593086,552.132836,652.989566,600.781421,545.022624,602.124953,529.775690,528.472127,480.642655,...,651.077936,522.605968,569.525160,535.361653,499.641618,587.586143,574.555595,600.538879,546.660930,549.851323


In [117]:
nci_df.to_csv('../../data_preprocessed/grn_input/count_matrix_nci.txt' ,sep = "\t")
nci_df

,352_120501,938_131101,564_120517,142_120419,456_120507,950_131107,812_130702,360_120502,283_120430,340_120501,...,588_120522,213_120425,174_120424,04_120405,508_120515,633_120524,146_120419,345_120501,498_120515,152_120419
ENSG00000101901,381.584859,415.660727,400.729313,359.696638,412.832101,393.279400,425.798569,369.168485,354.778728,529.577639,...,348.370890,575.077436,377.965207,415.768825,438.602684,342.797302,482.021145,355.230899,313.789066,368.087123
ENSG00000174839,396.591005,312.375334,450.384902,334.004021,360.408977,349.925765,360.746010,324.205657,284.726877,387.495833,...,406.274499,338.872097,402.205256,345.078823,346.574443,365.947249,311.510264,332.206675,401.186691,329.468146
ENSG00000151466,229.379662,236.800657,266.572108,214.533352,221.979166,227.090467,198.114612,235.857643,245.181478,357.049732,...,221.172800,239.077136,249.582726,252.995795,165.455030,200.336085,196.743325,239.013383,286.411496,185.853826
ENSG00000168288,360.147507,372.835076,376.337094,340.427175,391.535207,341.667930,328.219730,331.305051,353.648860,382.882787,...,309.452072,374.051616,339.360685,412.048298,390.630516,365.947249,488.579256,381.544299,294.835364,399.465041
ENSG00000112343,201.511105,166.264291,153.322520,160.578856,145.801814,91.868416,266.124106,183.795421,61.012902,182.676607,...,123.401133,163.692454,320.507313,129.288292,121.398957,139.790068,93.999588,78.940200,164.265417,195.508570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,49.305909,31.489449,27.005671,61.662281,20.477783,26.837964,106.449642,29.975219,12.428554,47.053065,...,16.137071,51.692354,30.524506,32.554606,31.328763,16.917269,33.883573,17.542267,54.755139,20.516331
ENSG00000100528,767.457188,753.227621,595.867066,709.116230,670.033053,831.976893,796.893850,640.523099,767.180753,830.348214,...,673.960033,997.949608,716.428112,793.402254,982.939944,617.925525,920.321552,787.209216,572.822993,1000.472868
ENSG00000116205,503.777763,435.813974,360.656382,486.875092,408.736545,353.022453,419.884700,347.081482,347.999517,319.222758,...,385.391230,368.308021,495.574333,410.188035,367.133943,380.193371,415.347019,367.291208,391.709841,299.297070
ENSG00000100603,604.533315,554.214303,574.088299,503.575293,578.292586,506.824633,563.296024,539.553940,595.440732,572.940267,...,557.203577,473.128627,565.601141,547.847512,490.490948,694.498429,489.672275,685.244791,501.220119,539.458832


In [118]:
nci_data = pd.read_csv('../../data_preprocessed/grn_input/count_matrix_nci.txt', sep="\t", index_col=0)
nci_data 

,352_120501,938_131101,564_120517,142_120419,456_120507,950_131107,812_130702,360_120502,283_120430,340_120501,...,588_120522,213_120425,174_120424,04_120405,508_120515,633_120524,146_120419,345_120501,498_120515,152_120419
ENSG00000101901,381.584859,415.660727,400.729313,359.696638,412.832101,393.279400,425.798569,369.168485,354.778728,529.577639,...,348.370890,575.077436,377.965207,415.768825,438.602684,342.797302,482.021145,355.230899,313.789066,368.087123
ENSG00000174839,396.591005,312.375334,450.384902,334.004021,360.408977,349.925765,360.746010,324.205657,284.726877,387.495833,...,406.274499,338.872097,402.205256,345.078823,346.574443,365.947249,311.510264,332.206675,401.186691,329.468146
ENSG00000151466,229.379662,236.800657,266.572108,214.533352,221.979166,227.090467,198.114612,235.857643,245.181478,357.049732,...,221.172800,239.077136,249.582726,252.995795,165.455030,200.336085,196.743325,239.013383,286.411496,185.853826
ENSG00000168288,360.147507,372.835076,376.337094,340.427175,391.535207,341.667930,328.219730,331.305051,353.648860,382.882787,...,309.452072,374.051616,339.360685,412.048298,390.630516,365.947249,488.579256,381.544299,294.835364,399.465041
ENSG00000112343,201.511105,166.264291,153.322520,160.578856,145.801814,91.868416,266.124106,183.795421,61.012902,182.676607,...,123.401133,163.692454,320.507313,129.288292,121.398957,139.790068,93.999588,78.940200,164.265417,195.508570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000120457,49.305909,31.489449,27.005671,61.662281,20.477783,26.837964,106.449642,29.975219,12.428554,47.053065,...,16.137071,51.692354,30.524506,32.554606,31.328763,16.917269,33.883573,17.542267,54.755139,20.516331
ENSG00000100528,767.457188,753.227621,595.867066,709.116230,670.033053,831.976893,796.893850,640.523099,767.180753,830.348214,...,673.960033,997.949608,716.428112,793.402254,982.939944,617.925525,920.321552,787.209216,572.822993,1000.472868
ENSG00000116205,503.777763,435.813974,360.656382,486.875092,408.736545,353.022453,419.884700,347.081482,347.999517,319.222758,...,385.391230,368.308021,495.574333,410.188035,367.133943,380.193371,415.347019,367.291208,391.709841,299.297070
ENSG00000100603,604.533315,554.214303,574.088299,503.575293,578.292586,506.824633,563.296024,539.553940,595.440732,572.940267,...,557.203577,473.128627,565.601141,547.847512,490.490948,694.498429,489.672275,685.244791,501.220119,539.458832
